In [1]:
import duckdb
import os

con = duckdb.connect()

DATA_DIR = r"data"
PROCESSED_DIR = r"data\processed"

raw_file = os.path.join(DATA_DIR, "option_prices_full.csv.gz")
parquet_file = os.path.join(PROCESSED_DIR, "option_prices_full.parquet")

print("Converting raw option file to parquet...")

con.execute(f"""
COPY (
    SELECT
        secid,
        CAST(date AS DATE) AS date,
        CAST(exdate AS DATE) AS exdate,
        cp_flag,
        strike_price,
        best_bid,
        best_offer,
        (best_bid + best_offer) / 2.0 AS mid,
        impl_volatility,
        delta,
        gamma,
        vega,
        theta,
        volume,
        open_interest,
        optionid,
        ticker,
        cusip,
        exercise_style,
        (CAST(exdate AS DATE) - CAST(date AS DATE)) AS dte
    FROM read_csv_auto('{raw_file}', compression='gzip')
    WHERE best_bid IS NOT NULL
      AND best_offer IS NOT NULL
      AND strike_price IS NOT NULL
)
TO '{parquet_file}' (FORMAT PARQUET)
""")

print("Saved:", parquet_file)

Converting raw option file to parquet...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved: data\processed\option_prices_full.parquet


In [2]:
con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    MIN(dte) AS min_dte,
    MAX(dte) AS max_dte
FROM read_parquet('{parquet_file}')
""").df()

,n_rows,min_dte,max_dte
0,414200044,0,120


In [3]:
earnings_prices_clean_file = os.path.join(PROCESSED_DIR, "earnings_prices_clean.parquet")
name_file = os.path.join(PROCESSED_DIR, "security_name_history_small.parquet")
crsp_file = os.path.join(PROCESSED_DIR, "crsp_daily_stock_small_v2.parquet")
sec_price_file = os.path.join(PROCESSED_DIR, "security_prices_small.parquet")

opt_short_file = os.path.join(PROCESSED_DIR, "option_prices_earnings_short_v2.parquet")

# ============================================================
# Helper
# ============================================================

def show(title, query, n=20):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)
    df = con.execute(query).df()
    if len(df) > n:
        print(df.head(n).to_string(index=False))
        print(f"\n[showing first {n} rows out of {len(df)}]")
    else:
        print(df.to_string(index=False))


# ============================================================
# STEP 1 — Load earnings events
# ============================================================

con.execute(f"""
CREATE OR REPLACE TEMP TABLE earnings_prices_clean AS
SELECT *
FROM read_parquet('{earnings_prices_clean_file}')
""")

# ============================================================
# STEP 2 — Build CRSP identifier history
# ============================================================

con.execute(f"""
CREATE OR REPLACE TEMP TABLE crsp_id_history AS
SELECT DISTINCT
    permno,
    date,
    NCUSIP AS ncusip,
    TICKER AS crsp_ticker
FROM read_parquet('{crsp_file}')
WHERE NCUSIP IS NOT NULL
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_crsp_id AS
SELECT
    e.*,
    c.ncusip,
    c.crsp_ticker
FROM earnings_prices_clean e
LEFT JOIN crsp_id_history c
  ON e.permno = c.permno
 AND e.prev_date = c.date
""")

# ============================================================
# STEP 3 — Map to OptionMetrics secid
# ============================================================

con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_name_history AS
SELECT
    secid,
    ticker AS om_ticker,
    cusip AS om_cusip,
    effect_date
FROM read_parquet('{name_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_secid AS
SELECT *
FROM (
    SELECT
        e.*,
        o.secid,
        o.om_ticker,
        o.om_cusip,
        o.effect_date,
        ROW_NUMBER() OVER (
            PARTITION BY e.gvkey, e.rdq
            ORDER BY o.effect_date DESC, o.secid
        ) AS rn
    FROM event_crsp_id e
    LEFT JOIN om_name_history o
      ON e.ncusip = o.om_cusip
     AND o.effect_date <= e.prev_date
) t
WHERE rn = 1
  AND secid IS NOT NULL
""")

# ============================================================
# STEP 4 — Ensure OM underlying price exists
# ============================================================

con.execute(f"""
CREATE OR REPLACE TEMP TABLE om_sec_prices AS
SELECT
    secid,
    date,
    close AS om_close
FROM read_parquet('{sec_price_file}')
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE event_ready AS
SELECT
    e.*
FROM event_secid e
JOIN om_sec_prices s
  ON e.secid = s.secid
 AND e.prev_date = s.date
WHERE s.om_close IS NOT NULL
""")

# ============================================================
# STEP 5 — Build secid/date pairs needed
# ============================================================

con.execute("""
CREATE OR REPLACE TEMP TABLE needed_secid_dates AS
SELECT DISTINCT
    secid,
    prev_date AS trade_date
FROM event_ready
""")

con.execute("""
CREATE OR REPLACE TEMP TABLE needed_secids AS
SELECT DISTINCT secid
FROM needed_secid_dates
""")

show(
    "Events ready",
    "SELECT COUNT(*) AS n_rows FROM event_ready"
)

# ============================================================
# STEP 6 — Build targeted option subset (0–21 DTE)
# ============================================================

con.execute(f"""
COPY (
    SELECT
        secid,
        date,
        exdate,
        cp_flag,
        strike_price,
        best_bid,
        best_offer,
        (best_bid + best_offer)/2.0 AS mid,
        impl_volatility,
        delta,
        gamma,
        vega,
        theta,
        volume,
        open_interest,
        optionid,
        ticker,
        cusip,
        exercise_style,
        exdate - date AS dte
    FROM read_parquet('{parquet_file}')
    WHERE secid IN (SELECT secid FROM needed_secids)
      AND (secid, date) IN (
            SELECT secid, trade_date
            FROM needed_secid_dates
      )
      AND exdate >= date
      AND (exdate - date) BETWEEN 0 AND 21
      AND cp_flag IN ('C','P')
      AND best_bid IS NOT NULL
      AND best_offer IS NOT NULL
) TO '{opt_short_file}' (FORMAT PARQUET)
""")

print("\nSaved:", opt_short_file)

# ============================================================
# STEP 7 — Diagnostics
# ============================================================

show(
    "Rows in new options subset",
    f"SELECT COUNT(*) AS n_rows FROM read_parquet('{opt_short_file}')"
)

show(
    "DTE distribution",
    f"""
    SELECT
        MIN(dte) AS min_dte,
        APPROX_QUANTILE(dte,0.01) AS p01,
        APPROX_QUANTILE(dte,0.5) AS median,
        APPROX_QUANTILE(dte,0.99) AS p99,
        MAX(dte) AS max_dte
    FROM read_parquet('{opt_short_file}')
    """
)

show(
    "Expiry coverage relative to earnings",
    f"""
    WITH option_chain AS (
        SELECT DISTINCT secid, date, exdate
        FROM read_parquet('{opt_short_file}')
    )
    SELECT
        SUM(CASE WHEN before_expiry=1 THEN 1 ELSE 0 END) AS events_with_before_expiry,
        SUM(CASE WHEN after_expiry=1 THEN 1 ELSE 0 END) AS events_with_after_expiry,
        COUNT(*) AS total_events
    FROM (
        SELECT
            e.gvkey,
            e.rdq,
            MAX(CASE WHEN c.exdate <= e.rdq THEN 1 ELSE 0 END) AS before_expiry,
            MAX(CASE WHEN c.exdate >  e.rdq THEN 1 ELSE 0 END) AS after_expiry
        FROM event_ready e
        LEFT JOIN option_chain c
          ON e.secid = c.secid
         AND e.prev_date = c.date
        GROUP BY e.gvkey,e.rdq
    ) x
    """
)

show(
    "Sample option rows",
    f"""
    SELECT
        secid,date,exdate,cp_flag,
        strike_price/1000.0 AS strike,
        best_bid,best_offer,mid,dte
    FROM read_parquet('{opt_short_file}')
    LIMIT 30
    """
)


Events ready
 n_rows
  12417


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Saved: data\processed\option_prices_earnings_short_v2.parquet

Rows in new options subset
 n_rows
2154081

DTE distribution
 min_dte  p01  median  p99  max_dte
       0    0      10   21       21

Expiry coverage relative to earnings
 events_with_before_expiry  events_with_after_expiry  total_events
                    1204.0                    9804.0         12417

Sample option rows
 secid       date     exdate cp_flag  strike  best_bid  best_offer    mid  dte
101062 2018-06-13 2018-06-15       C   170.0     84.65       87.70 86.175    2
101062 2018-06-13 2018-06-15       C   175.0     78.95       83.30 81.125    2
101062 2018-06-13 2018-06-15       C   180.0     73.85       78.35 76.100    2
101062 2018-06-13 2018-06-15       C   185.0     69.25       73.45 71.350    2
101062 2018-06-13 2018-06-15       C   190.0     64.30       66.70 65.500    2
101062 2018-06-13 2018-06-15       C   195.0     59.10       63.45 61.275    2
101062 2018-06-13 2018-06-15       C   200.0     55.50    